In [13]:
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import glob

class FeatureExtractor():
  def __init__(self, feature_type, n_features=1000):
      self.n_features = n_features
      self.extractor = self.create_extractor(feature_type)

  def create_extractor(self, feature_type):
    if feature_type == "SIFT":
      return cv2.SIFT_create(nfeatures=self.n_features)
    elif feature_type == "ORB":
      return cv2.ORB_create(nfeatures=self.n_features)
    elif feature_type == "KAZE":    #KAZE and AKAZE don't have a built-in nfeatures method
      return cv2.KAZE_create()
    elif feature_type == "AKAZE":
      return cv2.AKAZE_create()
    else:
      raise ValueError(f"Unsuported feature type: {feature_type}")

  def extract(self, frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    start_time = time.time()
    keypoints, descriptors = self.extractor.detectAndCompute(gray, None)
    end_time = time.time()

    execution_time = (end_time - start_time) * 1000  # conversion to ms

    return keypoints, descriptors, execution_time

In [8]:
class FeatureMatcher:
    def __init__(self, feature_type):
      self.match_metric = self.get_match_metric(feature_type)
      self.bf_matcher = cv2.BFMatcher(self.match_metric, crossCheck=False)

    def get_match_metric(self, feature_type):
      if feature_type in ["ORB","AKAZE"]:
        return cv2.NORM_HAMMING
      elif feature_type in ["SIFT","KAZE"]:
        return cv2.NORM_L2
      else:
        raise(ValueError(f"Unsuported feature type: {feature_type}"))

    def match(self, des1, des2, ratio=0.75):
      matches = self.bf_matcher.knnMatch(des1,des2,k=2)
      good = []
      for m,n in matches:
        if m.distance < ratio*n.distance:
          good.append([m])
      return good

In [18]:
def calibrate_camera(image_dir, chessboard_size=(9, 6), square_size_mm=25.0):
    objpoints = []
    imgpoints = []

    objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2)
    objp *= square_size_mm

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # Advanced flags to improve detection on screens
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE + cv2.CALIB_CB_FAST_CHECK

    images = glob.glob(f"./{image_dir}/*.jpeg")

    if not images:
        print("No images found.")
        return None, None

    successes = 0
    gray = None

    for fname in images:
        img = cv2.imread(fname)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        ret, corners = cv2.findChessboardCorners(gray, chessboard_size, flags)

        if ret:
            objpoints.append(objp)
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            imgpoints.append(corners2)
            successes += 1

    if successes == 0:
        print("Calibration failed: Chessboard not detected in any image.")
        return None, None

    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

    print(f"Calibration successful ({successes}/{len(images)} images used).")
    print(f"Intrinsic Matrix (K):\n{np.round(K, 2)}")
    print(f"Distortion Coefficients:\n{np.round(dist.flatten(), 5)}")

    total_error = 0
    for i in range(len(objpoints)):
        projected_imgpoints, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
        error = cv2.norm(imgpoints[i], projected_imgpoints, cv2.NORM_L2) / len(projected_imgpoints)
        total_error += error

    print(f"Mean Reprojection Error: {total_error / len(objpoints):.4f} px")

    return K, dist

In [43]:
class PlanarPoseEstimator:
    def __init__(self, K, dist_coefs=None):
        self.K = K
        self.dist_coefs = dist_coefs if dist_coefs is not None else np.zeros(5)

    def get_homography(self, src_pts, dst_pts):
        H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
        return H, mask

    def estimate_pose_manual(self, H):
        if H is None:
            return None

        # M = K^-1 * H
        K_inv = np.linalg.inv(self.K)
        M = K_inv @ H

        h1 = M[:, 0]
        h2 = M[:, 1]
        h3 = M[:, 2]

        # Scale factor
        norm_h1 = np.linalg.norm(h1)
        norm_h2 = np.linalg.norm(h2)
        lam = 2.0 / (norm_h1 + norm_h2)

        r1 = h1 * lam
        r2 = h2 * lam
        t = h3 * lam

        r3 = np.cross(r1, r2)

        R_approx = np.column_stack((r1, r2, r3))

        U, _, Vt = np.linalg.svd(R_approx)
        R = U @ Vt

        if np.linalg.det(R) < 0:
            Vt[2, :] *= -1
            R = U @ Vt

        if t[2] < 0:
            t = -t
            R[:, :2] = -R[:, :2]

        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = t

        return T

    def estimate_pose_cv2(self, H):
        if H is None:
            return None

        num_solutions, Rs, Ts, Ns = cv2.decomposeHomographyMat(H, self.K)

        valid_poses = []
        for i in range(num_solutions):
            R = Rs[i]
            t = Ts[i].flatten()

            if t[2] > 0:
                T = np.eye(4)
                T[:3, :3] = R
                T[:3, 3] = t
                valid_poses.append(T)

        return valid_poses[0] if valid_poses else None

In [44]:
#K_real, dist_real = calibrate_camera("images", chessboard_size=(5, 7), square_size_mm=15.0)

# Pre simulated values
K_real = np.array([[3.03743495e+03, 0.0, 1.54707905e+03],
                  [0.0, 3.03806327e+03, 2.01560818e+03],
                  [0.0, 0.0, 1.00000000e+00]])

dist_real = np.array([[1.29474053e-01, -3.98869600e-01,  1.37191037e-04,  8.03452690e-04, 8.00421825e-01]])

feature_type = "ORB"
extractor = FeatureExtractor(feature_type=feature_type)
matcher = FeatureMatcher(feature_type=feature_type)
pose_estimator = PlanarPoseEstimator(K_real, dist_real)

img_gabarito = cv2.imread("./img1.jpeg")
img_cena = cv2.imread("./img2.jpeg")

kp1, des1, _ = extractor.extract(img_gabarito)
kp2, des2, _ = extractor.extract(img_cena)

matches = matcher.match(des1, des2)

if len(matches) >= 4:
    src_pts = np.float32([kp1[m[0].queryIdx].pt for m in matches]).reshape(-1, 2)
    dst_pts = np.float32([kp2[m[0].trainIdx].pt for m in matches]).reshape(-1, 2)

    H, mask = pose_estimator.get_homography(src_pts, dst_pts)

    world_T_cam = pose_estimator.estimate_pose_manual(H)
    world_T_cam_cv = pose_estimator.estimate_pose_cv2(H)

    if world_T_cam is not None and world_T_cam_cv is not None:
        print(f'Camera Pose:\n{np.round(world_T_cam, 4)}')
        print(f'\nCamera Pose (cv2 function):\n{np.round(world_T_cam_cv, 4)}')
else:
    print("Matches insuficientes para calcular a Homografia.")

Camera Pose:
[[ 1.0150000e-01 -3.4000000e-03 -9.9480000e-01  5.1644200e+01]
 [ 9.5060000e-01  2.9530000e-01  9.5900000e-02  1.7081290e+02]
 [ 2.9340000e-01 -9.5540000e-01  3.3200000e-02  1.0542356e+03]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00]]

Camera Pose (cv2 function):
[[ 3.52900000e-01 -9.13400000e-01 -1.72900000e-01  1.44155000e+03]
 [-9.34100000e-01 -3.43700000e-01 -9.62000000e-02  4.76693230e+03]
 [ 2.88000000e-02  1.96300000e-01 -9.80100000e-01  2.94000686e+04]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
